In [1]:
import numpy as np
import polars as pl
import pandas as pd
import tensorflow as tf
import matplotlib.pyplot as plt
from sklearn.preprocessing import StandardScaler, MinMaxScaler
from working_data import clean_cols, clean_non_minute_rows, alt_label_df as label_df, normalize_by_window, split_df
from tensorflow.keras.callbacks import EarlyStopping, ModelCheckpoint
from model_builder_trans import combined_loss


2024-10-26 23:26:56.901544: I tensorflow/core/util/port.cc:113] oneDNN custom operations are on. You may see slightly different numerical results due to floating-point round-off errors from different computation orders. To turn them off, set the environment variable `TF_ENABLE_ONEDNN_OPTS=0`.
2024-10-26 23:26:56.938444: I tensorflow/core/platform/cpu_feature_guard.cc:210] This TensorFlow binary is optimized to use available CPU instructions in performance-critical operations.
To enable the following instructions: AVX2 AVX_VNNI FMA, in other operations, rebuild TensorFlow with the appropriate compiler flags.
2024-10-26 23:26:58.060181: W tensorflow/compiler/tf2tensorrt/utils/py_utils.cc:38] TF-TRT Warning: Could not find TensorRT
2024-10-26 23:26:59.584174: I external/local_xla/xla/stream_executor/cuda/cuda_executor.cc:984] could not open file to read NUMA node: /sys/bus/pci/devices/0000:01:00.0/numa_node
Your kernel may have been built without NUMA support.
2024-10-26 23:26:59.618060: 

In [2]:
NORMALIZING_WINDOW_SIZE = 60*3
LABELING_WINDOW_SIZE = 20
POSITIVE_SLOPE = 0.3
LABEL_CUR_CANDLE_MULTIPLIER = 0
LABEL_MEAN_MULTIPLIER = 5
BATCH_SIZE = 32
NUM_TOKENS = 128
LOOKBACK_WINDOW = NUM_TOKENS + 1
D_MODEL = 128
FF_DIM = 256
NUM_HEADS = D_MODEL//16

In [3]:
source_csv = "data/GBPUSD/minutes.csv"
working_path = "working"

In [4]:
def prepare_data(file_path, num_tokens, window_size=1440, batch_size=32, smote=False, shuffle=False, cols=['open', 'high', 'low', 'close']):
    scaler = MinMaxScaler()
    stand_scaler = StandardScaler()
    emd_range = window_size
    # Load CSV lazily with Polars
    collect_cols = cols + ['target']
    df_lazy = pl.scan_csv(file_path).select(collect_cols)
    
    # Collect the dataframe and determine total number of rows
    df_collected = df_lazy.collect()
    total_rows = df_collected.shape[0]
    if smote:
        df_collected = df_collected.with_columns(pl.arange(0, total_rows).alias("index"))
        indices_target_1 = df_collected.filter(
            (pl.col("target") == 1) & (pl.col("index") >= emd_range)
        ).select("index").to_series().to_list()

        # Get indices where target is 0 and >= num_tokens
        indices_target_0 = df_collected.filter(
            (pl.col("target") == 0) & (pl.col("index") >= emd_range)
        ).select("index").to_series().to_list()

        df_collected = df_collected.drop('index')
    
    while True:  # Loop to reshuffle and restart at each epoch
        # Create an array of indices to use for shuffling
        indices = list(range(emd_range, total_rows))
        if smote:
            indices_target_1_complete = []
            while len(indices_target_1_complete) < len(indices_target_0):
                indices_target_1_complete += indices_target_1

            indices_target_1_complete = indices_target_1_complete[:len(indices_target_0)]

            indices = indices_target_0 + indices_target_1_complete

        # Shuffle indices if required
        if shuffle:
            np.random.shuffle(indices)

        input_list = []
        target_list = []

        for idx in indices:
            #create imfs
            signal = np.array(df_collected[cols][idx - emd_range + 1:idx + 1])
            # signal = scaler.fit_transform(signal)

            # Fetch the previous `num_prev + 1` rows for the input based on the current index
            input_rows = signal[-num_tokens:]
            target_value = df_collected[idx, -1]  # Get 'target' for the target

            # Append the input rows to the input list
            input_list.append(input_rows)
            target_list.append(target_value)

            # Yield once we have enough for a batch
            if len(input_list) == batch_size:
                # Convert lists to NumPy arrays
                input_array = np.array(input_list)
                target_array = np.array(target_list)

                # Convert NumPy arrays to TensorFlow tensors
                input_tensor = tf.convert_to_tensor(input_array, dtype=tf.float32)
                target_tensor = tf.convert_to_tensor(target_array, dtype=tf.int32)

                yield input_tensor, target_tensor

                # Reset lists for the next batch

                input_list.clear()
                target_list.clear()

        break

In [5]:
def create_dataset_generator(file_path, batch_size, num_tokens, window_size=LOOKBACK_WINDOW, shuffle=False, repeat=False, smote=False, cols=['open', 'high', 'low', 'close']):
    dataset = tf.data.Dataset.from_generator(
        lambda: prepare_data(file_path, window_size=window_size, batch_size=batch_size, num_tokens=num_tokens, shuffle=shuffle, smote=smote, cols=cols),
        output_signature=(
            tf.TensorSpec(shape=(None, num_tokens, len(cols)), dtype=tf.float32),
            tf.TensorSpec(shape=(None,), dtype=tf.int32)
        )
    )
    if repeat:
        dataset = dataset.repeat()
    return dataset 

In [6]:
train_path = 'working/train.csv'
val_path = 'working/val.csv'
test_path = 'working/test.csv'

cols = [
    'open_normalized',
    'high_normalized',
    'low_normalized',
    'close_normalized'
]

train_dataset = create_dataset_generator(train_path, batch_size=BATCH_SIZE, num_tokens=NUM_TOKENS, repeat=True, shuffle=True, smote=True, cols=cols)
val_dataset = create_dataset_generator(val_path, batch_size=BATCH_SIZE, num_tokens=NUM_TOKENS, cols=cols)
test_dataset = create_dataset_generator(test_path, batch_size=BATCH_SIZE, num_tokens=NUM_TOKENS, cols=cols)


In [7]:
import tensorflow as tf
from tensorflow.keras.layers import Input, Conv1D, MaxPooling1D, Flatten, Dense, concatenate, LayerNormalization, Dropout, Lambda
from tensorflow.keras.models import Model
from tensorflow.keras.layers import MultiHeadAttention, Add, Embedding

input_length = NUM_TOKENS 

class PositionalEncoding(tf.keras.layers.Layer):
    def __init__(self, maxlen, d_model):
        super(PositionalEncoding, self).__init__()
        self.pos_encoding = self.positional_encoding(maxlen, d_model)

    def positional_encoding(self, maxlen, d_model):
        positions = np.arange(maxlen)[:, np.newaxis]
        angles = np.arange(d_model)[np.newaxis, :]
        angle_rates = 1 / np.power(10000, (2 * (angles // 2)) / np.float32(d_model))
        angle_rads = positions * angle_rates

        angle_rads[:, 0::2] = np.sin(angle_rads[:, 0::2])
        angle_rads[:, 1::2] = np.cos(angle_rads[:, 1::2])

        return tf.cast(angle_rads[np.newaxis, ...], dtype=tf.float32)

    def call(self, inputs):
        return inputs + self.pos_encoding[:, :tf.shape(inputs)[1], :]

class TransformerBlock(tf.keras.layers.Layer):
    def __init__(self, embed_dim, num_heads, ff_dim, rate=0.1):
        super(TransformerBlock, self).__init__()
        self.att = tf.keras.layers.MultiHeadAttention(num_heads=num_heads, key_dim=embed_dim)
        self.ffn = tf.keras.Sequential(
            [Dense(ff_dim, activation="relu"), Dense(embed_dim)]
        )
        self.layernorm1 = LayerNormalization(epsilon=1e-6)
        self.layernorm2 = LayerNormalization(epsilon=1e-6)
        self.dropout1 = Dropout(rate)
        self.dropout2 = Dropout(rate)

    def call(self, inputs, training):
        attn_output, attention_scores = self.att(inputs, inputs, return_attention_scores=True)
        attn_output = self.dropout1(attn_output, training=training)
        out1 = self.layernorm1(inputs + attn_output)
        ffn_output = self.ffn(out1)
        ffn_output = self.dropout2(ffn_output, training=training)
        return self.layernorm2(out1 + ffn_output), attention_scores  # Return output and attention scores

In [8]:
auc =  tf.keras.metrics.AUC()
auc.reset_state()
prec = tf.keras.metrics.Precision()
prec.reset_state()

In [9]:
# Define the model architecture
input_shape = (NUM_TOKENS, 4)
input_layer = Input(shape=input_shape)
x = Conv1D(filters=D_MODEL // 2, kernel_size=3, activation='relu', padding="same")(input_layer)
x = Conv1D(filters=D_MODEL, kernel_size=3, activation='relu', padding="same")(x)
x = MaxPooling1D(pool_size=2, padding="same")(x)
x = Dropout(0.1)(x)
x = PositionalEncoding(x.shape[1], d_model=D_MODEL)(x)

# Store attention scores
attention_scores_list = []
for _ in range(4):  # Assuming you want 4 transformer blocks
    x, attention_scores = TransformerBlock(D_MODEL, NUM_HEADS, FF_DIM)(x, training=True)
    attention_scores_list.append(attention_scores)

# Global pooling and output
x = tf.keras.layers.GlobalAveragePooling1D()(x)
outputs = Dense(1, activation='sigmoid')(x)
model = Model(inputs=input_layer, outputs=outputs)

model.load_weights("best_cnn_trans_model.keras", skip_mismatch=True)

learning_rate = 1e-3  # Adjust this value as needed
optimizer = tf.keras.optimizers.Adam(learning_rate=learning_rate)


# Compile the model
model.compile(
    optimizer=optimizer, 
    loss=combined_loss, 
    metrics=['accuracy',auc, prec])

model.summary()


Model: "functional_4"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ input_layer (InputLayer)        │ (None, 128, 4)         │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ conv1d (Conv1D)                 │ (None, 128, 64)        │           832 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ conv1d_1 (Conv1D)               │ (None, 128, 128)       │        24,704 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ max_pooling1d (MaxPooling1D)    │ (None, 64, 128)        │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout (Dropout)               │ (None, 64, 128)        │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ positional_encoding             │ (None, 64, 128)        │             0 │
│ (PositionalEncoding)            │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ transformer_block               │ [(None, 64, 128),      │       593,920 │
│ (TransformerBlock)              │ (None, 8, 64, 64)]     │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ transformer_block_1             │ [(None, 64, 128),      │       593,920 │
│ (TransformerBlock)              │ (None, 8, 64, 64)]     │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ transformer_block_2             │ [(None, 64, 128),      │       593,920 │
│ (TransformerBlock)              │ (None, 8, 64, 64)]     │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ transformer_block_3             │ [(None, 64, 128),      │       593,920 │
│ (TransformerBlock)              │ (None, 8, 64, 64)]     │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ global_average_pooling1d        │ (None, 128)            │             0 │
│ (GlobalAveragePooling1D)        │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_8 (Dense)                 │ (None, 1)              │           129 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 2,401,345 (9.16 MB)

 Trainable params: 2,401,345 (9.16 MB)

 Non-trainable params: 0 (0.00 B)

In [20]:
# Function to retrieve attention scores
def get_attention_scores(model, inputs):
    attention_scores_list = []
    
    # Define a new model that returns both outputs and attention scores
    for layer in model.layers:
        if isinstance(layer, TransformerBlock):
            x, scores = layer(inputs, training=False)  # Get output and attention scores
            attention_scores_list.append(scores)
            inputs = x  # Pass output to the next block
        else:
            inputs = layer(inputs)  # Pass through other layers

    return attention_scores_list

In [22]:
for features, labels in test_dataset.take(1):
    input_data = features

2024-10-26 23:35:05.810350: W tensorflow/core/framework/local_rendezvous.cc:404] Local rendezvous is aborting with status: OUT_OF_RANGE: End of sequence


In [23]:
attention_scores = get_attention_scores(model, input_data)

TypeError: too many positional arguments

In [11]:
import numpy as np
import matplotlib.pyplot as plt

# Assuming val_dataset is your validation dataset
def get_attention_scores(model, val_dataset):
    attention_scores_list = []

    for batch in val_dataset:
        x = batch[0]  # Assuming x is the first element of the batch
        attention_scores = []  # Reset for new batch

        # Run forward pass through the model
        for block in model.layers:
            if isinstance(block, TransformerBlock):
                x, scores = block(x, training=False)  # Get both output and scores
                attention_scores.append(scores)

        attention_scores_list.append(attention_scores)

    return attention_scores_list

# Get attention scores for the validation dataset
attention_scores = get_attention_scores(model, val_dataset)

# Example: Plot the attention scores for the first batch
def plot_attention_scores(scores):
    # Scores is a list of attention scores for each transformer block
    for i, score in enumerate(scores):
        plt.figure(figsize=(12, 6))
        plt.title(f'Attention Scores for Transformer Block {i + 1}')
        # Average scores across heads
        avg_scores = tf.reduce_mean(score, axis=1)  # Average over heads
        plt.imshow(avg_scores[0], aspect='auto', cmap='viridis')  # Show only the first example
        plt.colorbar()
        plt.xlabel("Key Positions")
        plt.ylabel("Query Positions")
        plt.show()

# Plot attention scores for the first batch of attention scores
plot_attention_scores(attention_scores[0])  # For the first validation batch


2024-10-26 23:27:49.584289: W tensorflow/core/framework/op_kernel.cc:1827] INVALID_ARGUMENT: required broadcastable shapes
2024-10-26 23:27:49.584341: W tensorflow/core/framework/local_rendezvous.cc:404] Local rendezvous is aborting with status: INVALID_ARGUMENT: required broadcastable shapes


InvalidArgumentError: Exception encountered when calling TransformerBlock.call().

[1m{{function_node __wrapped__AddV2_device_/job:localhost/replica:0/task:0/device:GPU:0}} required broadcastable shapes [Op:AddV2] name: [0m

Arguments received by TransformerBlock.call():
  • inputs=tf.Tensor(shape=(32, 128, 4), dtype=float32)
  • training=False

In [ ]:
for val_batch in val_dataset.take(1):  # Take one batch from the validation dataset
    inputs, _ = val_batch  # Assuming your val_dataset yields (inputs, labels)

2024-10-26 20:44:06.513435: W tensorflow/core/framework/local_rendezvous.cc:404] Local rendezvous is aborting with status: OUT_OF_RANGE: End of sequence


In [ ]:
for i, layer in enumerate(model.layers):
    print(f"Layer {i}: {layer.name} ({layer.__class__.__name__})")

Layer 0: input_layer (InputLayer)
Layer 1: conv1d (Conv1D)
Layer 2: conv1d_1 (Conv1D)
Layer 3: max_pooling1d (MaxPooling1D)
Layer 4: dropout (Dropout)
Layer 5: positional_encoding (PositionalEncoding)
Layer 6: transformer_block (TransformerBlock)
Layer 7: transformer_block_1 (TransformerBlock)
Layer 8: transformer_block_2 (TransformerBlock)
Layer 9: transformer_block_3 (TransformerBlock)
Layer 10: global_average_pooling1d (GlobalAveragePooling1D)
Layer 11: dense_8 (Dense)


In [ ]:
# Pass the entire `inputs` tensor to the Transformer block without iterating over it
transformer_output, attention_scores = model.layers[6].call(inputs, training=False, return_attention_scores=True)

# Print shapes to verify
print("Transformer output shape:", transformer_output.shape)
print("Attention scores shape:", attention_scores.shape)



TypeError: TransformerBlock.call() got an unexpected keyword argument 'return_attention_scores'

In [ ]:
print(inputs.shape)

(32, 128, 4)


In [ ]:
model.evaluate(val_dataset)

I0000 00:00:1729968256.439921   77516 service.cc:145] XLA service 0x7f7140014db0 initialized for platform CUDA (this does not guarantee that XLA will be used). Devices:
I0000 00:00:1729968256.439982   77516 service.cc:153]   StreamExecutor device (0): NVIDIA GeForce RTX 3070 Ti Laptop GPU, Compute Capability 8.6
2024-10-26 20:44:16.520982: I tensorflow/compiler/mlir/tensorflow/utils/dump_mlir_util.cc:268] disabling MLIR crash reproducer, set env var `MLIR_CRASH_REPRODUCER_DIRECTORY` to enable.
2024-10-26 20:44:16.690008: I external/local_xla/xla/stream_executor/cuda/cuda_dnn.cc:465] Loaded cuDNN version 8907
I0000 00:00:1729968258.438089   77672 asm_compiler.cc:369] ptxas warning : Registers are spilled to local memory in function 'triton_gemm_dot_37', 8 bytes spill stores, 8 bytes spill loads

I0000 00:00:1729968258.838674   77672 asm_compiler.cc:369] ptxas warning : Registers are spilled to local memory in function 'triton_gemm_dot_37', 16 bytes spill stores, 16 bytes spill loads

I0

     10/Unknown 8s 20ms/step - accuracy: 0.6827 - auc: 0.8318 - loss: 0.9955 - precision_1: 0.1788

I0000 00:00:1729968263.530962   77516 device_compiler.h:188] Compiled cluster using XLA!  This line is logged at most once for the lifetime of the process.


2100/2100 ━━━━━━━━━━━━━━━━━━━━ 49s 19ms/step - accuracy: 0.6879 - auc: 0.7702 - loss: 0.9408 - precision_1: 0.2315


2024-10-26 20:45:04.310657: W tensorflow/core/framework/local_rendezvous.cc:404] Local rendezvous is aborting with status: OUT_OF_RANGE: End of sequence
	 [[{{node IteratorGetNext}}]]
2024-10-26 20:45:04.310761: W tensorflow/core/framework/local_rendezvous.cc:404] Local rendezvous is aborting with status: OUT_OF_RANGE: End of sequence
	 [[{{node IteratorGetNext}}]]
	 [[IteratorGetNext/_4]]
2024-10-26 20:45:04.310774: I tensorflow/core/framework/local_rendezvous.cc:422] Local rendezvous recv item cancelled. Key hash: 8020434073634055103
2024-10-26 20:45:04.310800: I tensorflow/core/framework/local_rendezvous.cc:422] Local rendezvous recv item cancelled. Key hash: 5266965311296358320
/usr/lib/python3.10/contextlib.py:153: UserWarning: Your input ran out of data; interrupting training. Make sure that your dataset or generator can generate at least `steps_per_epoch * epochs` batches. You may need to use the `.repeat()` function when building your dataset.
  self.gen.throw(typ, value, trace

[0.9398537278175354,
 0.6880059242248535,
 0.7710319757461548,
 0.22696343064308167]

In [ ]:
model.predict(inputs)

1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 31ms/step


array([[0.7083509 ],
       [0.07478628],
       [0.00657972],
       [0.01863836],
       [0.7481126 ],
       [0.7534808 ],
       [0.7516744 ],
       [0.6611553 ],
       [0.7534603 ],
       [0.00767499],
       [0.00482882],
       [0.7534646 ],
       [0.7534098 ],
       [0.36850652],
       [0.29539096],
       [0.7532547 ],
       [0.7534803 ],
       [0.04038849],
       [0.10806444],
       [0.5230257 ],
       [0.75347555],
       [0.00464084],
       [0.75347525],
       [0.21191353],
       [0.24385715],
       [0.21331494],
       [0.46694854],
       [0.00603501],
       [0.74779224],
       [0.4750755 ],
       [0.7534716 ],
       [0.31365514]], dtype=float32)